In [ ]:
import shutil
from pathlib import Path
import pandas as pd
import json

from utils import nrcs_utils

### Assemble the test sites

In [ ]:
test_locs = [
    'nrcs-09361500:CO:USGS', # DRGC2 in CBRFC
    'nrcs-09109209:CO:USGS', # TPIC2 in CBRFC
    'nrcs-10128500:UT:USGS', # OAWU1 in CBRFC
]

### fetch NRCS timeseries data from API

In [ ]:
# test with single location
test_loc = test_locs[0]

In [ ]:
# get station information for the test location
station_info = nrcs_utils.get_stations(station_triplets=test_loc)

# display station information
print(json.dumps(station_info, indent=2))

In [ ]:
# get forecast information for the test location
forecast_data = nrcs_utils.get_forecasts(station_triplets=test_loc)
print(json.dumps(forecast_data, indent=2))

In [ ]:
forecast_data = nrcs_utils.get_forecasts(station_triplets=test_locs[1])
print(json.dumps(forecast_data, indent=2))

In [ ]:
# Build ingest-ready forecast rows for two test locations
import importlib
importlib.reload(nrcs_utils)

forecast_locations = [test_locs[0], test_locs[1]]

forecast_frames = []
for loc in forecast_locations:
    loc_forecasts = nrcs_utils.get_forecasts(station_triplets=loc)
    loc_df = nrcs_utils.forecasts_to_dataframe(loc_forecasts)
    forecast_frames.append(loc_df)

forecast_df = pd.concat(forecast_frames, ignore_index=True)

# Validate required schema and six rows per forecast data entry
print(forecast_df.columns.tolist())
rows_per_entry = (
    forecast_df.groupby(["location_id", "reference_time"], dropna=False)
    .size()
    .reset_index(name="n_rows")
)
print(rows_per_entry["n_rows"].value_counts().sort_index())

forecast_df.head(20)